In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

bucket = "data-engineering-project-sportsbar--dp-db-128"
base_folder = "canadian_market_new_three_linked"

if data_source in ["customers", "gross_price", "products"]:
    base_path = f"s3://{bucket}/{base_folder}/{data_source}.csv"

elif data_source in ["order", "orders"]:
    base_path = f"s3://{bucket}/{base_folder}/order/*.csv"

else:
    raise ValueError(f"Unknown data_source: {data_source}")

print(base_path)

s3://data-engineering-project-sportsbar--dp-db-128/canadian_market_new_three_linked/customers.csv


In [0]:
from pyspark.sql import functions as F
df = (

    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

# **Bronze Schema**

In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
display(df.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Toronto,2026-05-30T02:48:07.002Z,customers.csv,1393
789202,FitFuel Market,Vancouver,2026-05-30T02:48:07.002Z,customers.csv,1393
789203,FitFuel Market,Calgary,2026-05-30T02:48:07.002Z,customers.csv,1393
789301,Athlete's Choice Store,Toronto,2026-05-30T02:48:07.002Z,customers.csv,1393
789303,Athlete's Choice Store,Calgary,2026-05-30T02:48:07.002Z,customers.csv,1393
789101,Endurance Foods,Toronto,2026-05-30T02:48:07.002Z,customers.csv,1393
789102,Endurance Foods,Vancouver,2026-05-30T02:48:07.002Z,customers.csv,1393
789103,Endurance Foods,Calgary,2026-05-30T02:48:07.002Z,customers.csv,1393
789121,HydroBoost Nutrition,Vancouver,2026-05-30T02:48:07.002Z,customers.csv,1393
789122,HydroBoost Nutrition,Calgary,2026-05-30T02:48:07.002Z,customers.csv,1393


In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

# **Silver Schema**

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|  Toronto|2026-05-30 02:48:...|customers.csv|     1393|
|     789202|      FitFuel Market|Vancouver|2026-05-30 02:48:...|customers.csv|     1393|
|     789203|      FitFuel Market|  Calgary|2026-05-30 02:48:...|customers.csv|     1393|
|     789301|Athlete's Choice ...|  Toronto|2026-05-30 02:48:...|customers.csv|     1393|
|     789303|Athlete's Choice ...|  Calgary|2026-05-30 02:48:...|customers.csv|     1393|
|     789101|     Endurance Foods|  Toronto|2026-05-30 02:48:...|customers.csv|     1393|
|     789102|     Endurance Foods|Vancouver|2026-05-30 02:48:...|customers.csv|     1393|
|     789103|     Endurance Foods|  Calgary|2026-05-30 02:48:...|customers.csv|     1393|
|     7891

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

customer_id,count
789321,2
789503,2
789522,2
789603,2


In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

Rows before duplicates dropped:  39
Rows after duplicates dropped:  35


In [0]:

display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Vancouver,2026-05-30T02:48:14.887Z,customers.csv,1393
789401,SprintX nutrition,Toronto,2026-05-30T02:48:14.887Z,customers.csv,1393
789420,ZenAthlete foods,Ottawa,2026-05-30T02:48:14.887Z,customers.csv,1393
789421,ZenAthlete Foods,Vancouver,2026-05-30T02:48:14.887Z,customers.csv,1393
789521,PrimeFuel Nutrition,Ottawa,2026-05-30T02:48:14.887Z,customers.csv,1393
789702,StaminaX Store,Vancouver,2026-05-30T02:48:14.887Z,customers.csv,1393


In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:

# typos → correct names
city_mapping = {
    'Torono': 'Toronto',
    'Vancover': 'Vancouver',
    'Cagary': 'Calgary',
    'Otawa': 'Ottawa'
}


allowed = ["Toronto", "Vancouver", "Ottawa", "Calgary"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
df_silver.select('city').distinct().show()

+---------+
|     city|
+---------+
|  Toronto|
|Vancouver|
|  Calgary|
|   Ottawa|
+---------+



In [0]:
df_silver.select('customer_name').distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      FitFuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|HydroBoost Nutrition|
|MacroBite Superfoods|
|MacroBite superfoods|
|      PowerSnack Hub|
|      PowerSnack hub|
|   SprintX nutrition|
|   SprintX Nutrition|
|    ZenAthlete foods|
|    ZenAthlete Foods|
|Peak performance ...|
|Peak Performance ...|
| PrimeFuel Nutrition|
|       Recovery Lane|
|      StaminaX Store|
|EliteAthlete Nutr...|
|      GamePlan Foods|
|   Champion's choice|
+--------------------+
only showing top 20 rows


In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name")) #initcap = First letter capital, all other small
)

In [0]:
df_silver.select('customer_name').distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+



In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

None


In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("Canada"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size,customer,market,platform,channel
789201,Fitfuel Market,Toronto,2026-05-30T02:48:14.887Z,customers.csv,1393,Fitfuel Market-Toronto,Canada,Sports Bar,Acquisition
789202,Fitfuel Market,Vancouver,2026-05-30T02:48:14.887Z,customers.csv,1393,Fitfuel Market-Vancouver,Canada,Sports Bar,Acquisition
789203,Fitfuel Market,Calgary,2026-05-30T02:48:14.887Z,customers.csv,1393,Fitfuel Market-Calgary,Canada,Sports Bar,Acquisition
789301,Athlete's Choice Store,Toronto,2026-05-30T02:48:14.887Z,customers.csv,1393,Athlete's Choice Store-Toronto,Canada,Sports Bar,Acquisition
789303,Athlete's Choice Store,Calgary,2026-05-30T02:48:14.887Z,customers.csv,1393,Athlete's Choice Store-Calgary,Canada,Sports Bar,Acquisition
789101,Endurance Foods,Toronto,2026-05-30T02:48:14.887Z,customers.csv,1393,Endurance Foods-Toronto,Canada,Sports Bar,Acquisition
789102,Endurance Foods,Vancouver,2026-05-30T02:48:14.887Z,customers.csv,1393,Endurance Foods-Vancouver,Canada,Sports Bar,Acquisition
789103,Endurance Foods,Calgary,2026-05-30T02:48:14.887Z,customers.csv,1393,Endurance Foods-Calgary,Canada,Sports Bar,Acquisition
789121,Hydroboost Nutrition,Vancouver,2026-05-30T02:48:14.887Z,customers.csv,1393,Hydroboost Nutrition-Vancouver,Canada,Sports Bar,Acquisition
789122,Hydroboost Nutrition,Calgary,2026-05-30T02:48:14.887Z,customers.csv,1393,Hydroboost Nutrition-Calgary,Canada,Sports Bar,Acquisition


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# **Gold Schema**

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]